# Pipeline 2 — Deep Learning Model Development
**Sign Language Glove Sensor Classification**

This notebook covers every step from loading the preprocessed `.npz` dataset
through training and comparing three architectures:

| # | Architecture | Key idea |
|---|---|---|
| A | **Bidirectional LSTM** | Pure recurrent, full sequence context |
| B | **GRU** | Lighter recurrent, faster training |
| C | **1D CNN + LSTM** | Conv layers extract local patterns, LSTM captures long-range dynamics |

**Input shape per sample:** `(W=40, 22)` — 40 frames × 22 sensor channels  
**Channels:** flex curl [0–7] · pad z normalized [8–14] · pad r normalized [15–21]

---
## Cell 1 — Install & import dependencies

In [11]:
# Uncomment the line below if running for the first time in a fresh environment
# !pip install tensorflow scikit-learn numpy matplotlib seaborn --quiet

import os, json, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers, regularizers
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Reproducibility — fix every random seed that matters
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version : {tf.__version__}')
print(f'NumPy version      : {np.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available     : {len(gpus)}  {[g.name for g in gpus]}')

TensorFlow version : 2.21.0
NumPy version      : 2.3.3
GPUs available     : 0  []


---
## Cell 2 — Load preprocessed dataset

In [12]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PROCESSED_DIR   = '../data/processed'          # folder produced by preprocess_pipeline2.py
DATASET_PATH    = os.path.join(PROCESSED_DIR, 'extracted_features_deep_learning.npz')
METADATA_PATH   = os.path.join(PROCESSED_DIR, 'extracted_features_deep_learning_metadata.json')
ENCODER_PATH    = os.path.join("../models", 'label_encoder.pkl')

# ── Load arrays ───────────────────────────────────────────────────────────────
data = np.load(DATASET_PATH)
X_raw    = data['X_raw']     # (N_raw, W, 22)  — original recordings
y_raw    = data['y_raw']     # (N_raw,)
mask_raw = data['mask_raw']  # (N_raw, W)  bool
X_aug    = data['X_aug']     # (N_aug, W, 22)  — augmented copies
y_aug    = data['y_aug']     # (N_aug,)
mask_aug = data['mask_aug']  # (N_aug, W)  bool

# ── Load metadata ─────────────────────────────────────────────────────────────
with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    meta = json.load(f)

W          = meta['W']           # sequence length (frames)
N_FEATURES = meta['n_features']  # 22
N_CLASSES  = meta['n_classes']
CLASS_NAMES = meta['class_labels']

# ── Load label encoder (needed for decoding predictions later) ────────────────
with open(ENCODER_PATH, 'rb') as f:
    le: LabelEncoder = pickle.load(f)

print('Dataset loaded successfully')
print(f'  Window length W   : {W}')
print(f'  Features per frame: {N_FEATURES}')
print(f'  Classes           : {N_CLASSES}  → {CLASS_NAMES}')
print(f'  X_raw  : {X_raw.shape}  y_raw  : {y_raw.shape}')
print(f'  X_aug  : {X_aug.shape}  y_aug  : {y_aug.shape}')
print(f'  Global value range: [{data["X_raw"].min():.4f}, {data["X_raw"].max():.4f}]')

Dataset loaded successfully
  Window length W   : 40
  Features per frame: 22
  Classes           : 7  → ['احمد', 'انا', 'صوتك', 'عشري', 'فريق', 'مرحبا', 'من']
  X_raw  : (7, 40, 22)  y_raw  : (7,)
  X_aug  : (896, 40, 22)  y_aug  : (896,)
  Global value range: [0.0000, 1.0000]


---
## Cell 3 — Scaling check

All 22 sensor channels are already in **[0, 1]** after preprocessing:
- Flex curls were natively in [0, 1]
- Pad z was linearly mapped from [−1, 4] → [0, 1]
- Pad r was binned then scaled to [0, 1]

No further scaling is needed. This cell verifies it and provides a per-channel
breakdown so you can spot any channel that drifted out of range.

In [13]:
# Combine raw + aug for a full-dataset range check
X_all = np.concatenate([X_raw, X_aug], axis=0)  # (N_total, W, 22)

print(f'Global min  : {X_all.min():.6f}  (expect >= 0.0)')
print(f'Global max  : {X_all.max():.6f}  (expect <= 1.0)')
print()

# Per-channel statistics across all samples and timesteps
X_flat = X_all.reshape(-1, N_FEATURES)  # (N_total * W, 22)
feature_names = meta['feature_names']

print(f'{'Channel':<30}  {'Min':>7}  {'Max':>7}  {'Mean':>7}  {'Std':>7}  Status')
print('─' * 80)
all_ok = True
for i, name in enumerate(feature_names):
    ch = X_flat[:, i]
    cmin, cmax, cmean, cstd = ch.min(), ch.max(), ch.mean(), ch.std()
    ok = '✓' if (cmin >= -1e-5 and cmax <= 1 + 1e-5) else '✗ OUT OF RANGE'
    if '✗' in ok:
        all_ok = False
    print(f'{name:<30}  {cmin:7.4f}  {cmax:7.4f}  {cmean:7.4f}  {cstd:7.4f}  {ok}')

print()
if all_ok:
    print('✓ All channels within [0, 1].  No additional scaling required.')
else:
    print('✗ Some channels are out of range — investigate before training.')

Global min  : -0.000000  (expect >= 0.0)
Global max  : 1.000000  (expect <= 1.0)

Channel                             Min      Max     Mean      Std  Status
────────────────────────────────────────────────────────────────────────────────
flex_flex8_curl                 -0.0000   1.0000   0.1253   0.2795  ✓
flex_flex9_curl                 -0.0000   0.3631   0.0205   0.0455  ✓
flex_flex10_curl                 0.0000   1.0000   0.2291   0.2965  ✓
flex_flex11_curl                -0.0000   0.0409   0.0023   0.0046  ✓
flex_flex12_curl                -0.0000   1.0000   0.3638   0.4426  ✓
flex_flex13_curl                -0.0000   1.0000   0.3716   0.4472  ✓
flex_flex14_curl                -0.0000   0.0403   0.0022   0.0044  ✓
flex_flex15_curl                 0.0000   1.0000   0.4119   0.4547  ✓
pad_ch0_z_norm                   0.0000   1.0000   0.1587   0.3261  ✓
pad_ch1_z_norm                   0.0000   1.0000   0.0475   0.2114  ✓
pad_ch2_z_norm                   0.0000   0.0000   0.0000   0.

---
## Cell 4 — Train / Validation / Test split

**Strategy:**
- Split **raw** recordings first (70/15/15) so that real, un-augmented data
  is distributed across all three splits.
- Augmented copies of a recording always stay in the **same split** as their
  source. This prevents data leakage — the model must not be validated or
  tested on augmented versions of recordings it was trained on.
- All splits are stratified so every class appears in every split.

In [14]:
def build_splits(X_raw, y_raw, mask_raw, X_aug, y_aug, mask_aug,
                 val_frac=0.15, test_frac=0.15, seed=SEED):
    """
    Split raw samples stratified, then attach augmentations to the training split only.
    """
    N_raw = len(X_raw)
    aug_per_sample = len(X_aug) // N_raw   # how many aug copies per original
    indices = np.arange(N_raw)

    # Check if stratified split is feasible
    from collections import Counter
    counts = Counter(y_raw)
    min_count = min(counts.values())
    stratify = y_raw if min_count >= 3 else None
    if stratify is None:
        print(f'⚠  Only {min_count} sample(s) per class — falling back to '
               'non-stratified split.  Add more recordings for stratified splits.')

    # First split: train vs (val + test)
    idx_train, idx_temp = train_test_split(
        indices,
        test_size=val_frac + test_frac,
        stratify=stratify,
        random_state=seed
    )

    # Second split: val vs test from the temp pool
    stratify_temp = y_raw[idx_temp] if stratify is not None else None
    relative_test_frac = test_frac / (val_frac + test_frac)
    idx_val, idx_test = train_test_split(
        idx_temp,
        test_size=relative_test_frac,
        stratify=stratify_temp,
        random_state=seed
    )

    # For each raw training index, collect its augmented copies
    # Augmented samples are ordered: aug[i*K : (i+1)*K] belongs to raw[i]
    aug_idx_train = np.concatenate([
        np.arange(i * aug_per_sample, (i + 1) * aug_per_sample)
        for i in idx_train
    ])

    # Build final arrays: training = raw_train + all_aug_of_train
    X_train = np.concatenate([X_raw[idx_train], X_aug[aug_idx_train]], axis=0)
    y_train = np.concatenate([y_raw[idx_train], y_aug[aug_idx_train]], axis=0)
    m_train = np.concatenate([mask_raw[idx_train], mask_aug[aug_idx_train]], axis=0)

    X_val,  y_val,  m_val  = X_raw[idx_val],  y_raw[idx_val],  mask_raw[idx_val]
    X_test, y_test, m_test = X_raw[idx_test], y_raw[idx_test], mask_raw[idx_test]

    return (X_train, y_train, m_train, X_val,   y_val,   m_val, X_test,  y_test,  m_test)


(X_train, y_train, m_train, X_val,   y_val,   m_val, X_test,  y_test,  m_test) = build_splits(X_raw, y_raw, mask_raw, X_aug, y_aug, mask_aug)

# One-hot encode labels for categorical cross-entropy
Y_train = to_categorical(y_train, N_CLASSES)
Y_val   = to_categorical(y_val,   N_CLASSES)
Y_test  = to_categorical(y_test,  N_CLASSES)

print('Split summary')
print(f'  Train : {X_train.shape}  ({len(X_train)} samples)')
print(f'  Val   : {X_val.shape}   ({len(X_val)} samples)')
print(f'  Test  : {X_test.shape}   ({len(X_test)} samples)')
print()
print('Class distribution in each split:')
for split_name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    dist = {CLASS_NAMES[i]: int((y_split == i).sum()) for i in range(N_CLASSES)}
    print(f'  {split_name}: {dist}')

⚠  Only 1 sample(s) per class — falling back to non-stratified split.  Add more recordings for stratified splits.
Split summary
  Train : (516, 40, 22)  (516 samples)
  Val   : (1, 40, 22)   (1 samples)
  Test  : (2, 40, 22)   (2 samples)

Class distribution in each split:
  Train: {'احمد': 129, 'انا': 0, 'صوتك': 129, 'عشري': 129, 'فريق': 0, 'مرحبا': 0, 'من': 129}
  Val: {'احمد': 0, 'انا': 0, 'صوتك': 0, 'عشري': 0, 'فريق': 1, 'مرحبا': 0, 'من': 0}
  Test: {'احمد': 0, 'انا': 1, 'صوتك': 0, 'عشري': 0, 'فريق': 0, 'مرحبا': 1, 'من': 0}


---
## Cell 5 — Shared training configuration

All three architectures use identical:
- Optimizer, learning rate, gradient clipping
- Loss function
- Callbacks (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint)
- Batch size and max epochs

Only the model graph changes between architectures.

In [15]:
# ── Hyper-parameters ──────────────────────────────────────────────────────────
LEARNING_RATE  = 1e-3
BATCH_SIZE     = 32     # increase to 64 if GPU memory allows
MAX_EPOCHS     = 100
DROPOUT_RATE   = 0.3
L2_REG         = 1e-4   # L2 regularization weight on recurrent/dense layers
CLIP_NORM      = 1.0    # gradient clipping norm — prevents exploding gradients
                         # in recurrent layers on high-variance sensor sequences

# ── Optimizer ─────────────────────────────────────────────────────────────────
# clipnorm is set directly on the optimizer so it applies globally to every
# parameter regardless of which layer produces the gradient.
def make_optimizer():
    return optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=CLIP_NORM)


# ── Callbacks (same set for all three models) ─────────────────────────────────
def make_callbacks(model_name: str) -> list:
    """
    Returns a fresh list of callbacks for one training run.
    `model_name` is used for the checkpoint filename.
    """
    # Stop training when validation loss stops improving for 10 consecutive epochs.
    # restore_best_weights=True rewinds the model to the epoch with the lowest
    # val_loss once training ends — so you always get the best checkpoint.
    early_stop = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )

    # Halve the learning rate if val_loss doesn't improve for 5 epochs.
    # min_lr prevents the LR from shrinking to zero on long plateaus.
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )

    # Save the best model to disk so it can be loaded without retraining.
    checkpoint = callbacks.ModelCheckpoint(
        filepath=f'../models/best_{model_name}.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )

    # Log training metrics to CSV for offline analysis.
    csv_logger = callbacks.CSVLogger(f'../Results/Logs/training_log_{model_name}.csv', append=False)

    return [early_stop, reduce_lr, checkpoint, csv_logger]


# ── Shared compile helper ─────────────────────────────────────────────────────
def compile_model(model: keras.Model) -> None:
    """Compile in-place with the shared optimizer and loss."""
    model.compile(
        optimizer=make_optimizer(),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )


# ── Training runner ───────────────────────────────────────────────────────────
def train_model(model: keras.Model, model_name: str) -> keras.callbacks.History:
    """Compile, summarise, train, and return the History object."""
    compile_model(model)
    model.summary()
    print(f'\nTraining {model_name} …\n')
    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=make_callbacks(model_name),
        verbose=1
    )
    return history


# ── Evaluation helper ─────────────────────────────────────────────────────────
def evaluate_model(model: keras.Model, model_name: str,
                   history: keras.callbacks.History) -> dict:
    """
    Evaluate on the held-out test set, print the classification report,
    plot the confusion matrix, and return a results dict.
    """
    test_loss, test_acc = model.evaluate(X_test, Y_test, verbose=0)
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    print(f'\n── {model_name} Test Results ─────────────────────────')
    print(f'  Loss     : {test_loss:.4f}')
    print(f'  Accuracy : {test_acc * 100:.2f}%')
    print()
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES,
                                  zero_division=0))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(max(6, N_CLASSES), max(5, N_CLASSES - 1)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{model_name} — Confusion Matrix (Test Set)')
    plt.tight_layout()
    plt.savefig(f'../Results/Figures/confusion_{model_name}.png', dpi=120)
    plt.show()

    return {'name': model_name, 'test_loss': test_loss, 'test_acc': test_acc,
            'history': history}


# ── Learning curve plotter ────────────────────────────────────────────────────
def plot_learning_curves(history: keras.callbacks.History, model_name: str) -> None:
    """Plot training & validation loss and accuracy side by side."""
    h = history.history
    epochs_ran = range(1, len(h['loss']) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.plot(epochs_ran, h['loss'],     label='Train loss',  linewidth=2)
    ax1.plot(epochs_ran, h['val_loss'], label='Val loss',    linewidth=2, linestyle='--')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{model_name} — Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_ran, h['accuracy'],     label='Train acc',  linewidth=2)
    ax2.plot(epochs_ran, h['val_accuracy'], label='Val acc',    linewidth=2, linestyle='--')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title(f'{model_name} — Accuracy')
    ax2.set_ylim([0, 1.05])
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.suptitle(model_name, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'../Results/Figures/curves_{model_name}.png', dpi=120)
    plt.show()


print('Shared training config ready.')
print(f'  Optimizer     : Adam  lr={LEARNING_RATE}  clipnorm={CLIP_NORM}')
print(f'  Loss          : categorical_crossentropy')
print(f'  Batch size    : {BATCH_SIZE}')
print(f'  Max epochs    : {MAX_EPOCHS}  (EarlyStopping patience=10)')
print(f'  LR schedule   : ReduceLROnPlateau  factor=0.5  patience=5')
print(f'  Dropout rate  : {DROPOUT_RATE}')
print(f'  L2 reg        : {L2_REG}')

Shared training config ready.
  Optimizer     : Adam  lr=0.001  clipnorm=1.0
  Loss          : categorical_crossentropy
  Batch size    : 32
  Max epochs    : 100  (EarlyStopping patience=10)
  LR schedule   : ReduceLROnPlateau  factor=0.5  patience=5
  Dropout rate  : 0.3
  L2 reg        : 0.0001


---
## Cell 6 — Architecture A: Bidirectional LSTM

**Why Bidirectional?**  
We classify a complete, fixed-length window *after* collecting all W frames —
not one frame at a time. The backward LSTM pass therefore has access to
"future" frames relative to any given timestep, which helps the model
understand the full arc of a gesture (e.g. a finger that opens then closes).
This consistently outperforms unidirectional LSTM in offline gesture
classification tasks.

**Architecture:**
```
Input  (W, 22)
  → Masking                       ignore padding frames
  → BiLSTM(128, return_sequences=True)   first layer passes full sequence
  → Dropout(0.3)
  → BiLSTM(64, return_sequences=False)   second layer returns final state
  → Dropout(0.3)
  → Dense(256, relu, L2)
  → Dropout(0.3)
  → Dense(N_CLASSES, softmax)
```

The **Masking** layer tells Keras that a timestep with all-zero values is
padding. The LSTM gates skip those timesteps so padding never contributes
to the hidden state — critical for correctly processing short signs.

In [ ]:
def build_bilstm(W: int, n_features: int, n_classes: int) -> keras.Model:
    inputs = keras.Input(shape=(W, n_features), name='sensor_input')

    # Masking: timesteps where every feature == 0.0 are treated as padding.
    # This lets the LSTM ignore zero-padded frames at the end of short signs.
    x = layers.Masking(mask_value=0.0)(inputs)

    # First BiLSTM — return_sequences=True passes the full sequence (W, 256)
    # to the next layer so it can attend to all timesteps.
    # recurrent_dropout adds dropout inside the recurrent connections, which
    # is more effective for RNNs than plain dropout between layers alone.
    x = layers.Bidirectional(
        layers.LSTM(
            128,
            return_sequences=True,
            recurrent_dropout=0.1,
            kernel_regularizer=regularizers.l2(L2_REG)
        ),
        name='bilstm_1'
    )(x)
    x = layers.Dropout(DROPOUT_RATE)(x)

    # Second BiLSTM — return_sequences=False collapses the sequence to a
    # single (batch, 128) context vector representing the whole gesture.
    x = layers.Bidirectional(
        layers.LSTM(
            64,
            return_sequences=False,
            recurrent_dropout=0.1,
            kernel_regularizer=regularizers.l2(L2_REG)
        ),
        name='bilstm_2'
    )(x)
    x = layers.Dropout(DROPOUT_RATE)(x)

    # Dense classifier head
    x = layers.Dense(
        256, activation='relu',
        kernel_regularizer=regularizers.l2(L2_REG),
        name='dense_1'
    )(x)
    x = layers.Dropout(DROPOUT_RATE)(x)

    outputs = layers.Dense(n_classes, activation='softmax', name='output')(x)

    return keras.Model(inputs, outputs, name='BiLSTM')


model_bilstm  = build_bilstm(W, N_FEATURES, N_CLASSES)
history_bilstm = train_model(model_bilstm, 'BiLSTM')
plot_learning_curves(history_bilstm, 'BiLSTM')
results_bilstm = evaluate_model(model_bilstm, 'BiLSTM', history_bilstm)

---
## Cell 7 — Architecture B: GRU

**Why GRU vs LSTM?**  
GRU merges the LSTM's cell state and hidden state into a single hidden state
and uses only 2 gates (reset, update) instead of 3 (input, forget, output).
This means ~30% fewer parameters and faster training, with accuracy typically
within 1–2% of LSTM on sequences under 100 frames — which is our case.

Use GRU when:
- Training time matters more than squeezing the last 1% accuracy
- The dataset is smaller and LSTM risks overfitting due to its extra capacity

**Architecture:**
```
Input  (W, 22)
  → Masking
  → Bidirectional GRU(128, return_sequences=True)
  → Dropout(0.3)
  → Bidirectional GRU(64, return_sequences=False)
  → Dropout(0.3)
  → Dense(256, relu, L2)
  → Dropout(0.3)
  → Dense(N_CLASSES, softmax)
```

In [ ]:
def build_bigru(W: int, n_features: int, n_classes: int) -> keras.Model:
    inputs = keras.Input(shape=(W, n_features), name='sensor_input')

    x = layers.Masking(mask_value=0.0)(inputs)

    # GRU uses reset_after=True (the default in Keras) which is compatible
    # with cuDNN acceleration on GPU — important for larger datasets.
    x = layers.Bidirectional(
        layers.GRU(
            128,
            return_sequences=True,
            recurrent_dropout=0.1,
            kernel_regularizer=regularizers.l2(L2_REG),
            reset_after=True
        ),
        name='bigru_1'
    )(x)
    x = layers.Dropout(DROPOUT_RATE)(x)

    x = layers.Bidirectional(
        layers.GRU(
            64,
            return_sequences=False,
            recurrent_dropout=0.1,
            kernel_regularizer=regularizers.l2(L2_REG),
            reset_after=True
        ),
        name='bigru_2'
    )(x)
    x = layers.Dropout(DROPOUT_RATE)(x)

    x = layers.Dense(
        256, activation='relu',
        kernel_regularizer=regularizers.l2(L2_REG),
        name='dense_1'
    )(x)
    x = layers.Dropout(DROPOUT_RATE)(x)

    outputs = layers.Dense(n_classes, activation='softmax', name='output')(x)

    return keras.Model(inputs, outputs, name='BiGRU')


model_bigru  = build_bigru(W, N_FEATURES, N_CLASSES)
history_bigru = train_model(model_bigru, 'BiGRU')
plot_learning_curves(history_bigru, 'BiGRU')
results_bigru = evaluate_model(model_bigru, 'BiGRU', history_bigru)

---
## Cell 8 — Architecture C: 1D CNN + LSTM

**Why this hybrid?**  
Raw LSTM has to learn both *local* micro-patterns (e.g. a finger bending over
3–5 frames) and *global* sequence dynamics (the whole gesture arc) at the same
time.  The 1D CNN layers act as a **learned feature extractor** over short
temporal windows, replacing the hand-crafted rolling-mean and
direction-reversal features from Pipeline 1.  The LSTM then only needs to
model the longer-range dependencies across the CNN output — a simpler task.

This hybrid typically:
- Trains **faster** than pure LSTM (shorter sequence into the LSTM after pooling)
- **Generalises better** with limited data (CNN filters are parameter-efficient)
- Is more **robust to slight temporal misalignment** of a sign in the window

**Architecture:**
```
Input  (W=40, 22)
  → Conv1D(64,  kernel=3, relu, same padding)
  → BatchNorm
  → Conv1D(128, kernel=3, relu, same padding)
  → BatchNorm
  → MaxPool1D(2)              40 → 20 frames into LSTM
  → Dropout(0.3)
  → LSTM(128, return_sequences=False)
  → Dropout(0.3)
  → Dense(256, relu, L2)
  → Dropout(0.3)
  → Dense(N_CLASSES, softmax)
```

> **Note on Masking:**  `Conv1D` and `MaxPool1D` do not propagate Keras masks,
> so the Masking layer is omitted here.  Instead, BatchNormalization and the
> convolutional filters make the padded zero-frames contribute minimally to
> the feature maps.  For longer sequences with many padding frames, you can
> manually pass a `sample_weight` mask to `model.fit()` — not needed at our
> current sequence length.

In [ ]:
def build_cnn_lstm(W: int, n_features: int, n_classes: int) -> keras.Model:
    inputs = keras.Input(shape=(W, n_features), name='sensor_input')

    # ── CNN feature extraction block ──────────────────────────────────────────
    # kernel_size=3 looks at 3 consecutive frames at a time — enough to capture
    # the micro-dynamics of a finger curl or pad contact event.
    # 'same' padding keeps the time dimension unchanged so MaxPool halves it cleanly.
    x = layers.Conv1D(
        filters=64, kernel_size=3, padding='same', activation='relu',
        kernel_regularizer=regularizers.l2(L2_REG),
        name='conv1'
    )(inputs)
    # BatchNormalization stabilizes training by normalizing activations within
    # each mini-batch — especially helpful with small datasets.
    x = layers.BatchNormalization()(x)

    x = layers.Conv1D(
        filters=128, kernel_size=3, padding='same', activation='relu',
        kernel_regularizer=regularizers.l2(L2_REG),
        name='conv2'
    )(x)
    x = layers.BatchNormalization()(x)

    # MaxPool halves the time dimension (40 → 20 or W → W//2).
    # The LSTM now processes a shorter, richer sequence — fewer timesteps,
    # more abstract features — which speeds convergence significantly.
    x = layers.MaxPooling1D(pool_size=2, name='maxpool')(x)
    x = layers.Dropout(DROPOUT_RATE)(x)

    # ── LSTM temporal modelling block ─────────────────────────────────────────
    # Unidirectional LSTM here (not Bi) because the CNN already captures
    # local patterns in both directions via its receptive field.
    # A single LSTM layer suffices after the CNN has compressed the sequence.
    x = layers.LSTM(
        128,
        return_sequences=False,
        recurrent_dropout=0.1,
        kernel_regularizer=regularizers.l2(L2_REG),
        name='lstm'
    )(x)
    x = layers.Dropout(DROPOUT_RATE)(x)

    # ── Classifier head ───────────────────────────────────────────────────────
    x = layers.Dense(
        256, activation='relu',
        kernel_regularizer=regularizers.l2(L2_REG),
        name='dense_1'
    )(x)
    x = layers.Dropout(DROPOUT_RATE)(x)

    outputs = layers.Dense(n_classes, activation='softmax', name='output')(x)

    return keras.Model(inputs, outputs, name='CNN_LSTM')


model_cnn_lstm  = build_cnn_lstm(W, N_FEATURES, N_CLASSES)
history_cnn_lstm = train_model(model_cnn_lstm, 'CNN_LSTM')
plot_learning_curves(history_cnn_lstm, 'CNN_LSTM')
results_cnn_lstm = evaluate_model(model_cnn_lstm, 'CNN_LSTM', history_cnn_lstm)

---
## Cell 9 — Side-by-side comparison of all three architectures

In [ ]:
all_results = [results_bilstm, results_bigru, results_cnn_lstm]

# ── Summary table ──────────────────────────────────────────────────────────────
print('=' * 60)
print(f'  {'Model':<15}  {'Test Loss':>10}  {'Test Acc':>10}  {'Epochs run':>10}')
print('=' * 60)
for r in all_results:
    epochs_ran = len(r['history'].history['loss'])
    print(f"  {r['name']:<15}  {r['test_loss']:>10.4f}  "
          f"{r['test_acc']*100:>9.2f}%  {epochs_ran:>10d}")
print('=' * 60)

# ── Learning curves overlay ────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2196F3', '#4CAF50', '#FF5722']

for r, color in zip(all_results, colors):
    h = r['history'].history
    ep = range(1, len(h['loss']) + 1)
    ax1.plot(ep, h['val_loss'],     label=r['name'], color=color, linewidth=2)
    ax2.plot(ep, h['val_accuracy'], label=r['name'], color=color, linewidth=2)

ax1.set_xlabel('Epoch'); ax1.set_ylabel('Validation Loss')
ax1.set_title('Validation Loss — All Models')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation Accuracy')
ax2.set_title('Validation Accuracy — All Models')
ax2.set_ylim([0, 1.05])
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_all_models.png', dpi=120)
plt.show()

# ── Parameter count comparison ─────────────────────────────────────────────────
print('\nParameter counts:')
for model, name in [(model_bilstm, 'BiLSTM'),
                     (model_bigru,  'BiGRU'),
                     (model_cnn_lstm, 'CNN_LSTM')]:
    total = model.count_params()
    trainable = sum(tf.size(v).numpy() for v in model.trainable_variables)
    print(f'  {name:<15}: {total:>10,} total params')

---
## Cell 10 — Save the best model and run inference example

The `ModelCheckpoint` callback already saved a `.keras` file during training.
This cell demonstrates how to load it and run a prediction on a single sample
— exactly the call pattern your inference script will use.

In [ ]:
# ── Pick the best model by test accuracy ──────────────────────────────────────
best = max(all_results, key=lambda r: r['test_acc'])
print(f"Best model: {best['name']}  (test acc = {best['test_acc']*100:.2f}%)")

# ── Load from the saved checkpoint (proves the file is valid) ─────────────────
best_model = keras.models.load_model(f"best_{best['name']}.keras")
print(f"Loaded best_{best['name']}.keras successfully")

# ── Single-sample inference demo ──────────────────────────────────────────────
# Take one sample from the test set and predict it
sample_idx   = 0
sample_input = X_test[sample_idx : sample_idx + 1]   # shape (1, W, 22)
true_label   = CLASS_NAMES[y_test[sample_idx]]

probs      = best_model.predict(sample_input, verbose=0)[0]   # (N_CLASSES,)
pred_idx   = np.argmax(probs)
pred_label = CLASS_NAMES[pred_idx]
confidence = probs[pred_idx]

print(f'\nSingle-sample inference:')
print(f'  True label   : {true_label}')
print(f'  Predicted    : {pred_label}  (confidence {confidence*100:.1f}%)')
print(f'  Correct      : {"✓" if pred_label == true_label else "✗"}')
print()
print('Full probability distribution:')
for cls, p in sorted(zip(CLASS_NAMES, probs), key=lambda x: -x[1]):
    bar = '█' * int(p * 30)
    print(f'  {cls:<12} {p*100:5.1f}%  {bar}')

# ── Save the label encoder alongside the model for deployment ─────────────────
import shutil
shutil.copy(ENCODER_PATH, f"label_encoder_{best['name']}.pkl")
print(f"\nLabel encoder copied → label_encoder_{best['name']}.pkl")
print('These two files are everything needed for inference:')
print(f"  best_{best['name']}.keras")
print(f"  label_encoder_{best['name']}.pkl")